# Task 13 — Sentiment Analysis with a Custom Subword Tokenizer & Custom Embeddings

This notebook builds a complete NLP pipeline **from scratch**:

1. A custom **Byte-Pair Encoding (BPE)** subword tokenizer (no external tokenizer libraries)
2. Custom **word embeddings** trained with skip-gram + negative sampling (pure numpy)
3. A **text classifier** built on top of the custom tokenizer + custom embeddings
4. A **comparison** against a pre-trained tokenizer + pre-trained embeddings (GloVe)

Dataset: `Sentiment_Analysis.csv` — 80,000 balanced positive/negative texts.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from bpe_tokenizer import BPETokenizer
from train_embeddings import (
    build_vocab_and_corpus, train_word2vec, save_vectors_txt
)
from visualize_embeddings import load_vectors_txt, pca_2d

df = pd.read_csv("Sentiment_Analysis.csv")
print(df.shape)
df.head()


## 1. Train the custom BPE subword tokenizer

In [ ]:
sample = df.sample(n=4000, random_state=42).reset_index(drop=True)
corpus = sample["text"].astype(str).tolist()

tokenizer = BPETokenizer()
tokenizer.train(corpus, vocab_size=1500, min_pair_freq=3)
tokenizer.save("subword_vocab.json")


### Tokenize 10 sample sentences

In [ ]:
SAMPLE_SENTENCES = [
    "The movie was absolutely wonderful and heartwarming.",
    "This film was a complete waste of time, terribly boring.",
    "I loved the acting, the story was captivating from start to finish!",
    "Worst movie I have ever seen, the plot made no sense at all.",
    "A masterpiece of modern cinema, beautifully directed and acted.",
    "The special effects were amazing but the dialogue felt awkward.",
    "I would not recommend this film to anyone, very disappointing.",
    "An emotional rollercoaster with brilliant performances throughout.",
    "The pacing was slow and the characters were poorly developed.",
    "Absolutely thrilling from beginning to end, a must watch!",
]

for i, sent in enumerate(SAMPLE_SENTENCES, 1):
    print(f"[{i}] {sent}")
    print("   ->", tokenizer.tokenize(sent), "\n")


## 2. Train custom word embeddings (skip-gram + negative sampling, from scratch)

Implemented with plain numpy — no gensim training routines are used here.

In [ ]:
tokenized_docs = [tokenizer.tokenize(t) for t in sample["text"].astype(str)]

token2id, id_corpus, freqs = build_vocab_and_corpus(tokenized_docs, max_vocab=1800, min_freq=2)
id2token = {i: t for t, i in token2id.items()}
print(f"Word2Vec vocabulary size: {len(token2id)}; training sequences: {len(id_corpus)}")

EMBED_DIM = 50
model = train_word2vec(
    id_corpus, vocab_size=len(token2id), embed_dim=EMBED_DIM,
    window=2, negatives=5, epochs=2, lr=0.025, freqs=freqs
)

save_vectors_txt("custom_embeddings.vec", id2token, model.W_in)


### Sanity check: nearest neighbours in the learned embedding space

In [ ]:
def most_similar(token, topn=5):
    if token not in token2id:
        return []
    vecs = model.W_in
    norm_vecs = vecs / (np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-9)
    idx = token2id[token]
    sims = norm_vecs @ norm_vecs[idx]
    top_idx = np.argsort(-sims)[1:topn + 1]
    return [(id2token[i], float(sims[i])) for i in top_idx]

for probe in ["good</w>", "bad</w>", "movie</w>", "love</w>"]:
    print(f"{probe}: {most_similar(probe)}")


### Visualize embeddings with PCA

In [ ]:
tokens, vectors = load_vectors_txt("custom_embeddings.vec")

highlight_words = [
    "good</w>", "great</w>", "bad</w>", "terrible</w>", "horrible</w>",
    "love</w>", "hate</w>", "amazing</w>", "boring</w>", "wonderful</w>",
    "worst</w>", "best</w>", "happy</w>", "sad</w>", "awful</w>",
    "movie</w>", "film</w>", "show</w>", "story</w>", "acting</w>",
    "the</w>", "and</w>", "was</w>", "is</w>", "not</w>",
]
present = [w for w in highlight_words if w in tokens]
idx = [tokens.index(w) for w in present]

rng = np.random.default_rng(0)
other_idx = [i for i in range(len(tokens)) if i not in idx]
sample_other = rng.choice(other_idx, size=min(120, len(other_idx)), replace=False)

all_idx = list(idx) + list(sample_other)
coords = pca_2d(vectors[all_idx])
sub_tokens = [tokens[i] for i in all_idx]

plt.figure(figsize=(10, 8))
plt.scatter(coords[len(idx):, 0], coords[len(idx):, 1], c="lightgray", s=15, label="other tokens")
plt.scatter(coords[:len(idx), 0], coords[:len(idx), 1], c="crimson", s=40, label="highlighted words")
for i, tok in enumerate(sub_tokens[:len(idx)]):
    plt.annotate(tok.replace("</w>", ""), (coords[i, 0], coords[i, 1]), fontsize=9)
plt.title("PCA Projection of Custom-Trained Subword Embeddings")
plt.legend()
plt.tight_layout()
plt.show()


## 3. Text classification: custom pipeline vs. pre-trained pipeline

Both pipelines use the exact same downstream classifier (Logistic Regression)
so the only difference is the tokenizer + embeddings used to build document
vectors (average pooling of token embeddings).

In [ ]:
import re
import gensim.downloader as api
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

RANDOM_STATE = 42
SAMPLE_SIZE = 4000

def load_data():
    d = pd.read_csv("Sentiment_Analysis.csv").dropna(subset=["text", "sentiment"]).drop_duplicates(subset=["text"])
    per_class = SAMPLE_SIZE // 2
    parts = [g.sample(per_class, random_state=RANDOM_STATE) for _, g in d.groupby("sentiment")]
    return pd.concat(parts).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

data = load_data()
train_df, test_df = train_test_split(data, test_size=0.2, random_state=RANDOM_STATE, stratify=data["sentiment"])
print(train_df.shape, test_df.shape)


In [ ]:
# ---- Custom pipeline: BPE tokenizer + custom embeddings ----
def load_vectors_txt2(path):
    vecs = {}
    with open(path) as f:
        header = f.readline().split()
        dim = int(header[1])
        for line in f:
            parts = line.rstrip("\n").split(" ")
            vecs[parts[0]] = np.array(parts[1:], dtype=np.float64)
    return vecs, dim

custom_vecs, custom_dim = load_vectors_txt2("custom_embeddings.vec")

def doc_vector_custom(text):
    toks = tokenizer.tokenize(text)
    v = [custom_vecs[t] for t in toks if t in custom_vecs]
    return np.mean(v, axis=0) if v else np.zeros(custom_dim)

X_train_custom = np.vstack([doc_vector_custom(t) for t in train_df["text"]])
X_test_custom = np.vstack([doc_vector_custom(t) for t in test_df["text"]])

clf_custom = LogisticRegression(max_iter=1000)
clf_custom.fit(X_train_custom, train_df["sentiment"])
preds_custom = clf_custom.predict(X_test_custom)


In [ ]:
# ---- Pre-trained pipeline: regex word tokenizer + GloVe-50d ----
glove_model = api.load("glove-wiki-gigaword-50")
glove_dim = glove_model.vector_size
WORD_RE = re.compile(r"[a-zA-Z]+")

def pretrained_tokenize(text):
    return WORD_RE.findall(text.lower())

def doc_vector_pretrained(text):
    toks = pretrained_tokenize(text)
    v = [glove_model[t] for t in toks if t in glove_model]
    return np.mean(v, axis=0) if v else np.zeros(glove_dim)

X_train_glove = np.vstack([doc_vector_pretrained(t) for t in train_df["text"]])
X_test_glove = np.vstack([doc_vector_pretrained(t) for t in test_df["text"]])

clf_glove = LogisticRegression(max_iter=1000)
clf_glove.fit(X_train_glove, train_df["sentiment"])
preds_glove = clf_glove.predict(X_test_glove)


## 4. Evaluate & compare

In [ ]:
def evaluate(name, y_true, y_pred):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1-Score": f1_score(y_true, y_pred),
    }

results = [
    evaluate("Custom BPE + Custom Embeddings", test_df["sentiment"], preds_custom),
    evaluate("Pre-trained Tokenizer + GloVe-50d", test_df["sentiment"], preds_glove),
]
results_df = pd.DataFrame(results).set_index("Model").round(4)
results_df


In [ ]:
ax = results_df.plot(kind="bar", figsize=(9, 6), rot=15)
ax.set_title("Custom Pipeline vs. Pre-trained Pipeline")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, preds) in zip(axes, [("Custom BPE + Custom Embeddings", preds_custom),
                                     ("Pre-trained Tokenizer + GloVe-50d", preds_glove)]):
    cm = confusion_matrix(test_df["sentiment"], preds)
    ax.imshow(cm, cmap="Blues")
    ax.set_title(name, fontsize=9)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["neg", "pos"]); ax.set_yticklabels(["neg", "pos"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                     color="white" if cm[i, j] > cm.max()/2 else "black")
plt.tight_layout()
plt.show()


## Conclusion

- The **custom BPE tokenizer** successfully learns meaningful subword units purely from the training corpus's character statistics, with no external tokenizer libraries.
- The **custom skip-gram embeddings**, trained from scratch with numpy, learn real semantic structure (e.g. "movie" and "film" end up as close neighbours, and sentiment words cluster separately from stopwords in the PCA projection).
- On the downstream classification task, the **pre-trained GloVe pipeline outperforms the custom pipeline** — expected, since GloVe was trained on billions of words of text with a 400,000-word vocabulary, while the custom embeddings were trained from scratch on only a few thousand documents in a couple of epochs.
- This gap illustrates the value of pre-trained embeddings for small datasets, while still demonstrating that a subword tokenizer + embedding pipeline built entirely from scratch can learn genuine linguistic structure.
